# Hands-on AI Agent: Daily Sync & Team Dispatcher
## Orchestration menggunakan LangChain + Gemini + Google Sheets

**Use case:** Agent membaca project tracker dari Google Sheets, menganalisis progres dan risiko pekerjaan, menyusun action items, lalu membuat **draf email lokal** untuk direview manusia.

### Tujuan pembelajaran
1. Menghubungkan Gemini dengan LangChain.
2. Mengubah fungsi Python menjadi LangChain Tools menggunakan `@tool`.
3. Membuat agent dengan `create_agent()`.
4. Mengamati orchestration: model → tool → tool result → model.
5. Membuat draf email tanpa mengirim email.

### Alur
`Google Sheets → Python Tools → LangChain Agent (Gemini) → Tool Execution → Daily Sync Analysis → Local Email Draft`

**Penting:** Notebook ini tidak mengirim email. Draf hanya disimpan di memori runtime Colab dan ditampilkan untuk direview. Google Sheets dibaca ulang setiap kali tool dipanggil agar agent tidak menggunakan snapshot lama.

**Sebelum mulai:** siapkan Gemini API key dari Google AI Studio dan simpan di Colab Secrets dengan nama `GOOGLE_API_KEY`. Pastikan akun Google yang digunakan memiliki akses baca ke spreadsheet.

## 0. Konfigurasi awal

Notebook ini menggunakan ID spreadsheet dari kode notebook sumber. Jika menggunakan spreadsheet lain, ganti `SPREADSHEET_ID` pada cell konfigurasi.

Worksheet default adalah tab pertama. Nama tab dapat diganti setelah melihat daftar worksheet.

## 1. Instalasi library

In [1]:
%pip -q install -U langchain langchain-google-genai gspread pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


## 2. Autentikasi Google Colab dan konfigurasi Google Sheets

In [2]:
from google.colab import auth, userdata
from google.auth import default
import gspread
import pandas as pd
import json
from typing import Optional

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

# ID dari notebook sumber. Ganti jika memakai spreadsheet lain.
SPREADSHEET_ID = "1Wn1QVN5PmjminBDSnJtEnuKmvqSZSUpMGAMczu-e2ME"

spreadsheet = gc.open_by_key(SPREADSHEET_ID)
worksheet_names = [ws.title for ws in spreadsheet.worksheets()]

print("Spreadsheet:", spreadsheet.title)
print("Worksheet tersedia:", worksheet_names)

# Default ke worksheet pertama. Ubah jika tracker ada di tab lain.
WORKSHEET_NAME = worksheet_names[0]
worksheet = spreadsheet.worksheet(WORKSHEET_NAME)
print("Worksheet aktif:", WORKSHEET_NAME)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Spreadsheet: Project Management
Worksheet tersedia: ['Project Tracker']
Worksheet aktif: Project Tracker


## 3. Definisikan skema kolom dan fungsi membaca tracker

Fungsi ini dipanggil langsung oleh tool. Dengan demikian, data diambil ulang dari Google Sheets setiap kali agent meminta data.

In [3]:
REQUIRED_COLUMNS = [
    "Task ID", "Project", "Department", "Task / Deliverable", "Owner",
    "Priority", "Status", "Progress (%)", "Start Date", "Due Date",
    "Days to Due", "Overdue?", "Last Update", "Blocker / Notes"
]

def load_project_tracker() -> pd.DataFrame:
    """Ambil data terbaru dari worksheet Google Sheets."""
    records = worksheet.get_all_records()
    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError("Worksheet kosong atau tidak memiliki baris data.")

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(
            "Kolom wajib tidak ditemukan: " + ", ".join(missing) +
            ". Periksa header pada Google Sheets atau sesuaikan REQUIRED_COLUMNS."
        )

    # Normalisasi kolom teks
    text_cols = [
        "Task ID", "Project", "Department", "Task / Deliverable",
        "Owner", "Priority", "Status", "Overdue?", "Blocker / Notes"
    ]
    for col in text_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    # Normalisasi angka progress
    df["Progress (%)"] = pd.to_numeric(
        df["Progress (%)"].astype(str).str.replace("%", "", regex=False),
        errors="coerce"
    )

    # Konversi nilai kosong/NaN agar aman diserialisasi
    df = df.astype(object).where(pd.notna(df), None)
    return df

tasks = load_project_tracker()
print(f"Berhasil membaca {len(tasks)} task.")
display(tasks.head(10))

Berhasil membaca 300 task.


,Task ID,Project,Department,Task / Deliverable,Owner,Priority,Status,Progress (%),Start Date,Due Date,Days to Due,Overdue?,Last Update,Blocker / Notes
0,TASK-001,Sales Performance Analytics,Data & Analytics,Mengumpulkan kebutuhan data,Budi Santoso,Medium,In Progress,79,05-Sep-2026,23-Nov-2026,24,No,24-Okt-2026,Koordinasi lintas tim berlangsung
1,TASK-002,PMO Transformation,Project Management,Menyusun project charter,Citra Amelia,Critical,In Progress,35,28-Agu-2026,03-Nov-2026,4,No,27-Okt-2026,Sedang dikerjakan
2,TASK-003,Support Process Improvement,Customer Success,Memantau SLA respons,Rizky Maulana,Medium,Completed,100,01-Agu-2026,04-Nov-2026,5,No,25-Okt-2026,Selesai sesuai rencana
3,TASK-004,Data Quality Initiative,Data & Analytics,Membersihkan dataset operasional,Kevin Hartono,Medium,Completed,100,13-Agu-2026,20-Nov-2026,21,No,29-Okt-2026,Dokumentasi telah diperbarui
4,TASK-005,Service Delivery Improvement,Operations,Memeriksa checklist operasional,Aldi Wijaya,Low,In Review,91,07-Sep-2026,15-Nov-2026,16,No,29-Okt-2026,Menunggu review stakeholder
5,TASK-006,Enterprise Process Improvement,Project Management,Mengidentifikasi risiko proyek,Kevin Hartono,High,In Progress,45,11-Agu-2026,22-Okt-2026,-8,Yes,25-Okt-2026,Koordinasi lintas tim berlangsung
6,TASK-007,Internal Workflow Automation,Product & Engineering,Menulis unit test,Putri Ananda,Low,In Progress,19,27-Agu-2026,05-Des-2026,36,No,28-Okt-2026,Koordinasi lintas tim berlangsung
7,TASK-008,Digital Marketing Optimization,Marketing & Communications,Menyiapkan materi kampanye,Dimas Saputra,Medium,Completed,100,18-Sep-2026,09-Des-2026,40,No,27-Okt-2026,Selesai sesuai rencana
8,TASK-009,Enterprise Process Improvement,Project Management,Menyiapkan rapat steering committee,Fajar Nugroho,Low,In Progress,50,04-Sep-2026,22-Nov-2026,23,No,23-Okt-2026,Koordinasi lintas tim berlangsung
9,TASK-010,Operational Excellence,Operations,Menyelesaikan tiket internal,Aldi Wijaya,Medium,In Progress,43,03-Sep-2026,26-Nov-2026,27,No,24-Okt-2026,Sedang dikerjakan


## 4. Pemeriksaan data awal

Gunakan pandas untuk memahami kualitas data sebelum agent bekerja. Pemeriksaan ini bersifat deterministik, bukan kesimpulan LLM.

In [4]:
print("Jumlah task:", len(tasks))
print("\nDistribusi status:")
display(tasks["Status"].value_counts(dropna=False).rename_axis("Status").to_frame("Jumlah"))

print("\nDistribusi prioritas:")
display(tasks["Priority"].value_counts(dropna=False).rename_axis("Priority").to_frame("Jumlah"))

print("\nTask ID duplikat:", int(tasks["Task ID"].duplicated().sum()))
print("Task tanpa owner:", int((tasks["Owner"].fillna("").astype(str).str.strip() == "").sum()))
print("Task tanpa blocker/notes:", int((tasks["Blocker / Notes"].fillna("").astype(str).str.strip() == "").sum()))

display(tasks[[
    "Task ID", "Project", "Task / Deliverable", "Owner", "Priority",
    "Status", "Progress (%)", "Due Date", "Days to Due", "Overdue?", "Blocker / Notes"
]].head(10))

Jumlah task: 300

Distribusi status:


,Jumlah
Status,
In Progress,127
Completed,60
Not Started,52
In Review,43
Blocked,18



Distribusi prioritas:


,Jumlah
Priority,
Medium,157
High,78
Low,39
Critical,26



Task ID duplikat: 0
Task tanpa owner: 0
Task tanpa blocker/notes: 0


,Task ID,Project,Task / Deliverable,Owner,Priority,Status,Progress (%),Due Date,Days to Due,Overdue?,Blocker / Notes
0,TASK-001,Sales Performance Analytics,Mengumpulkan kebutuhan data,Budi Santoso,Medium,In Progress,79,23-Nov-2026,24,No,Koordinasi lintas tim berlangsung
1,TASK-002,PMO Transformation,Menyusun project charter,Citra Amelia,Critical,In Progress,35,03-Nov-2026,4,No,Sedang dikerjakan
2,TASK-003,Support Process Improvement,Memantau SLA respons,Rizky Maulana,Medium,Completed,100,04-Nov-2026,5,No,Selesai sesuai rencana
3,TASK-004,Data Quality Initiative,Membersihkan dataset operasional,Kevin Hartono,Medium,Completed,100,20-Nov-2026,21,No,Dokumentasi telah diperbarui
4,TASK-005,Service Delivery Improvement,Memeriksa checklist operasional,Aldi Wijaya,Low,In Review,91,15-Nov-2026,16,No,Menunggu review stakeholder
5,TASK-006,Enterprise Process Improvement,Mengidentifikasi risiko proyek,Kevin Hartono,High,In Progress,45,22-Okt-2026,-8,Yes,Koordinasi lintas tim berlangsung
6,TASK-007,Internal Workflow Automation,Menulis unit test,Putri Ananda,Low,In Progress,19,05-Des-2026,36,No,Koordinasi lintas tim berlangsung
7,TASK-008,Digital Marketing Optimization,Menyiapkan materi kampanye,Dimas Saputra,Medium,Completed,100,09-Des-2026,40,No,Selesai sesuai rencana
8,TASK-009,Enterprise Process Improvement,Menyiapkan rapat steering committee,Fajar Nugroho,Low,In Progress,50,22-Nov-2026,23,No,Koordinasi lintas tim berlangsung
9,TASK-010,Operational Excellence,Menyelesaikan tiket internal,Aldi Wijaya,Medium,In Progress,43,26-Nov-2026,27,No,Sedang dikerjakan


## 5. Buat Tool 1 — membaca project tracker

Dekorator `@tool` mengubah fungsi Python menjadi tool yang dapat dipilih model. Deskripsi docstring membantu model memahami kapan dan bagaimana tool digunakan.

Tool membaca ulang Google Sheets saat dipanggil dan mendukung filter status, owner, priority, project, department, dan overdue.

In [5]:
from langchain.tools import tool

@tool
def read_project_tracker_tool(
    status: Optional[str] = None,
    owner: Optional[str] = None,
    priority: Optional[str] = None,
    project: Optional[str] = None,
    department: Optional[str] = None,
    overdue_only: bool = False
) -> str:
    """Baca data terbaru dari Google Sheets project tracker.

    Gunakan tool ini sebelum membuat analisis daily sync.
    Filter opsional: status, owner, priority, project, department,
    atau overdue_only=True untuk hanya mengambil task overdue.
    """
    df = load_project_tracker()

    filters = [
        ("Status", status),
        ("Owner", owner),
        ("Priority", priority),
        ("Project", project),
        ("Department", department),
    ]

    for column, value in filters:
        if value:
            df = df[
                df[column].astype(str).str.casefold() ==
                str(value).strip().casefold()
            ]

    if overdue_only:
        overdue_values = {"yes", "ya", "true", "1"}
        df = df[df["Overdue?"].astype(str).str.casefold().isin(overdue_values)]

    records = df.to_dict(orient="records")
    return json.dumps(records, ensure_ascii=False, default=str)

# Uji tool secara langsung sebelum agent digunakan
tracker_preview = read_project_tracker_tool.invoke({})
preview_records = json.loads(tracker_preview)

print("Jumlah task yang dikembalikan tool:", len(preview_records))
print("Contoh record:")
print(json.dumps(preview_records[:1], ensure_ascii=False, indent=2))

Jumlah task yang dikembalikan tool: 300
Contoh record:
[
  {
    "Task ID": "TASK-001",
    "Project": "Sales Performance Analytics",
    "Department": "Data & Analytics",
    "Task / Deliverable": "Mengumpulkan kebutuhan data",
    "Owner": "Budi Santoso",
    "Priority": "Medium",
    "Status": "In Progress",
    "Progress (%)": 79,
    "Start Date": "05-Sep-2026",
    "Due Date": "23-Nov-2026",
    "Days to Due": 24,
    "Overdue?": "No",
    "Last Update": "24-Okt-2026",
    "Blocker / Notes": "Koordinasi lintas tim berlangsung"
  }
]


## 6. Buat Tool 2 — menyimpan draf email lokal

Tool ini **tidak mengirim email** dan tidak terhubung ke Gmail. Penerima ditetapkan oleh pengguna di konfigurasi, bukan dipilih bebas oleh model. Model hanya menyusun subject dan body.

In [6]:
# Ganti alamat ini sebelum menjalankan agent.
TEAM_EMAIL = "team@example.com"

email_drafts = []

@tool
def create_email_draft_tool(subject: str, body: str) -> str:
    """Simpan draf email daily sync secara lokal untuk direview manusia.
    Tool ini tidak mengirim email. Penerima menggunakan TEAM_EMAIL
    yang dikonfigurasi oleh pengguna.
    """
    if not TEAM_EMAIL or TEAM_EMAIL == "team@example.com":
        raise ValueError(
            "Silakan atur TEAM_EMAIL terlebih dahulu pada cell konfigurasi."
        )

    draft = {
        "to": TEAM_EMAIL,
        "subject": subject,
        "body": body,
        "status": "DRAFT_ONLY — belum dikirim"
    }
    email_drafts.append(draft)

    return json.dumps({
        "ok": True,
        "message": "Draf disimpan di runtime notebook. Tidak ada email yang dikirim.",
        "draft_number": len(email_drafts),
        "recipient": TEAM_EMAIL
    }, ensure_ascii=False)

print("Tool email siap. Penerima yang dikonfigurasi:", TEAM_EMAIL)

Tool email siap. Penerima yang dikonfigurasi: team@example.com


## 7. Siapkan Gemini API Key dan model LangChain

Simpan API key pada **Colab Secrets** dengan nama `GOOGLE_API_KEY`. Jika Secrets tidak tersedia, notebook akan meminta key secara tersembunyi.

Model default di bawah dapat diganti sesuai model Gemini yang tersedia pada akun/API key Anda.

In [13]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = None

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Masukkan GOOGLE_API_KEY: ")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY belum diisi.")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Ganti model jika model ini tidak tersedia untuk API key Anda.
MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.2
)

# Tes koneksi sederhana
test_response = llm.invoke("Balas hanya dengan: Gemini terhubung.")
print(test_response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Gemini terhubung.', 'extras': {'signature': 'EmAKXgFpFH0Tb+d5pTXuHhHi+sweUemSLV+u43Pi8D0UwmtrImi4AVUVGObMfZISNVjySyDCyqDBH0bDPgTwwmIjr3tTX9ELQvvyHuiLofKO+1qp2U2EPoZbjwWD6SIVynE='}}]


## 8. Daftarkan tools ke LangChain

Sekarang fungsi Python sudah menjadi LangChain Tools. LangChain membaca nama, deskripsi, dan schema argumen tool untuk membantu model menentukan tool yang relevan.

In [14]:
agent_tools = [
    read_project_tracker_tool,
    create_email_draft_tool
]

for registered_tool in agent_tools:
    print("-", registered_tool.name, ":", registered_tool.description.split("\n")[0])

- read_project_tracker_tool : Baca data terbaru dari Google Sheets project tracker.
- create_email_draft_tool : Simpan draf email daily sync secara lokal untuk direview manusia.


## 9. Bangun agent dan orchestration dengan `create_agent()`

`create_agent()` membangun agent graph yang mengatur siklus:
1. Model menerima instruksi dan konteks.
2. Model memutuskan apakah perlu memanggil tool.
3. LangChain menjalankan tool.
4. Hasil tool dikembalikan kepada model.
5. Model melanjutkan sampai menghasilkan jawaban akhir.

Kita tidak perlu menulis `execute_tool()` dan `run_agent()` secara manual seperti pada implementasi function calling sebelumnya.

In [15]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
Kamu adalah Daily Sync & Team Dispatcher untuk tim profesional.

SUMBER KEBENARAN
- Google Sheets project tracker adalah sumber kebenaran status pekerjaan.
- Selalu panggil read_project_tracker_tool sebelum membuat kesimpulan.
- Jangan mengarang task, owner, status, progress, tanggal, atau blocker.
- Jika data kosong atau tidak konsisten, nyatakan keterbatasannya.

ANALISIS
- Prioritaskan task Critical dan High.
- Soroti task overdue, task yang mendekati deadline, dan task berstatus Blocked.
- Bedakan blocker yang tercatat pada tracker dari risiko yang kamu simpulkan.
- Susun executive summary, progres, perhatian utama, dan action items per owner.
- Saat menyebut pekerjaan, selalu sertakan Task ID dan nama tugas:
  [TASK-ID] Nama Task.
- Sertakan project, owner, priority, status, progress, dan due date bila tersedia.

DRAF EMAIL
- Setelah analisis selesai, buat subject dan body email daily sync.
- Panggil create_email_draft_tool untuk menyimpan draf.
- Tool tersebut hanya menyimpan draf lokal, bukan mengirim email.
- Jangan pernah mengklaim email sudah dikirim.

GAYA
- Gunakan bahasa Indonesia yang ramah, ringkas, profesional, dan mudah dipindai.
- Jangan menyimpulkan pekerjaan selesai hanya berdasarkan asumsi.
"""

agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=SYSTEM_PROMPT,
    name="daily_sync_team_dispatcher"
)

print("Agent berhasil dibuat.")

Agent berhasil dibuat.


## 10. Jalankan Daily Sync Agent

Atur penerima email pada cell Tool 2 sebelum menjalankan cell ini. Prompt meminta agent membaca tracker terbaru, menganalisis pekerjaan, lalu menyimpan draf lokal.

In [17]:
# Set a custom TEAM_EMAIL to bypass the default placeholder validation
globals()["TEAM_EMAIL"] = "faransina.rumere@gmail.com"

request = """
Siapkan daily sync berdasarkan project tracker terbaru.

Buat:
1. Executive summary.
2. Ringkasan progres pekerjaan.
3. Daftar task Critical dan High.
4. Task overdue dan task yang deadline-nya paling dekat.
5. Task Blocked dan blocker yang tercatat.
6. Action items berdasarkan owner, lengkap dengan Task ID.

Setelah analisis selesai, siapkan draf email daily sync.
Jangan mengirim email.
"""

result = agent.invoke({
    "messages": [
        {"role": "user", "content": request}
    ]
})

final_message = result["messages"][-1]
print(final_message.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Berikut adalah hasil analisis *Daily Sync* dan dispatch tim berdasarkan data terbaru dari Google Sheets *project tracker*:\n\n---\n\n### 1. Executive Summary\nSecara umum, kegiatan operasional dan proyek tim berjalan dengan baik; banyak task yang telah mencapai status *Completed* atau *In Review*. Namun, perhatian manajerial mendesak diperlukan untuk menangani **task berstatus Blocked** serta **task yang sudah Overdue** (terutama dengan prioritas *Critical* dan *High*). Eskalasi lintas departemen diperlukan untuk menghapus hambatan (*blocker*) terkait dependensi vendor dan persetujuan manajemen.\n\n---\n\n### 2. Ringkasan Progres Pekerjaan\n* **Total Task Tercatat:** 300 task di berbagai departemen.\n* **Status Penyelesaian:** Sejumlah besar task telah selesai (*Completed*) atau berada di tahap akhir (*In Review*).\n* **Area Perhatian:** Sejumlah task mengalami penundaan (*overdue*) dari tenggat waktu aslinya dan beberapa task tertahan (*blocked*) karena menu

## 11. Amati orchestration dan tool calls

Cell ini menampilkan jejak pesan yang dikembalikan agent. Perhatikan:
- `AIMessage` dengan `tool_calls`: model meminta tool dijalankan.
- `ToolMessage`: hasil eksekusi tool.
- `AIMessage` berikutnya: model memproses hasil tool atau memberi jawaban akhir.

Urutan pesan aktual dapat berbeda tergantung keputusan model.

In [18]:
from langchain_core.messages import AIMessage, ToolMessage

for idx, message in enumerate(result["messages"], start=1):
    print("\n" + "=" * 72)
    print(f"MESSAGE {idx}: {type(message).__name__}")

    if isinstance(message, AIMessage) and message.tool_calls:
        for call in message.tool_calls:
            print("TOOL CALL:", call["name"])
            print("ARGUMENTS:", call["args"])

    elif isinstance(message, ToolMessage):
        print("TOOL NAME:", message.name)
        print("TOOL OUTPUT:")
        print(str(message.content)[:3000])

    else:
        content = message.content
        print("CONTENT:")
        print(content if isinstance(content, str) else str(content)[:3000])


MESSAGE 1: HumanMessage
CONTENT:

Siapkan daily sync berdasarkan project tracker terbaru.

Buat:
1. Executive summary.
2. Ringkasan progres pekerjaan.
3. Daftar task Critical dan High.
4. Task overdue dan task yang deadline-nya paling dekat.
5. Task Blocked dan blocker yang tercatat.
6. Action items berdasarkan owner, lengkap dengan Task ID.

Setelah analisis selesai, siapkan draf email daily sync.
Jangan mengirim email.


MESSAGE 2: AIMessage
TOOL CALL: read_project_tracker_tool
ARGUMENTS: {}

MESSAGE 3: ToolMessage
TOOL NAME: read_project_tracker_tool
TOOL OUTPUT:
[{"Task ID": "TASK-001", "Project": "Sales Performance Analytics", "Department": "Data & Analytics", "Task / Deliverable": "Mengumpulkan kebutuhan data", "Owner": "Budi Santoso", "Priority": "Medium", "Status": "In Progress", "Progress (%)": 79, "Start Date": "05-Sep-2026", "Due Date": "23-Nov-2026", "Days to Due": 24, "Overdue?": "No", "Last Update": "24-Okt-2026", "Blocker / Notes": "Koordinasi lintas tim berlangsung"}, 

## 12. Review draf email — tidak ada email terkirim

Draf yang dibuat hanya berada di runtime Colab. Periksa penerima, subject, isi, dan konsistensi Task ID sebelum digunakan.

In [19]:
if not email_drafts:
    print("Belum ada draf. Periksa apakah agent berhasil memanggil create_email_draft_tool.")
else:
    for i, draft in enumerate(email_drafts, start=1):
        print("=" * 72)
        print(f"DRAF {i}")
        print("To:", draft["to"])
        print("Subject:", draft["subject"])
        print("Status:", draft["status"])
        print("\n" + draft["body"])

DRAF 1
To: faransina.rumere@gmail.com
Subject: Daily Sync & Action Items - Progres & Task Prioritas
Status: DRAFT_ONLY — belum dikirim

Halo Tim,

Berikut adalah rangkuman Daily Sync berdasarkan Google Sheets project tracker terbaru.

1. EXECUTIVE SUMMARY
Secara keseluruhan, progres tim berjalan dengan baik dengan banyak task yang sudah berstatus Completed atau In Review. Namun, terdapat perhatian khusus pada beberapa task berprioritas Critical dan High yang berstatus Blocked maupun Overdue, yang memerlukan eskalasi dan tindakan cepat dari owner terkait.

2. RINGKASAN PROGRES PEKERJAAN
- Total Task Tercatat: 300 task
- Completed: Sejumlah task telah diselesaikan sesuai target di berbagai departemen.
- In Progress / In Review: Mayoritas task berjalan sesuai rencana atau menunggu review stakeholder.
- Blocked & Overdue: Memerlukan intervensi segera untuk mencegah dampak lanjutan pada timeline proyek.

3. DAFTAR TASK CRITICAL DAN HIGH
Berikut adalah daftar task dengan prioritas Critical d

## 13. Eksperimen untuk peserta

Jalankan ulang agent dengan pertanyaan berikut. Bandingkan tools yang dipanggil dan kualitas jawabannya.

1. “Tampilkan hanya task Critical.”
2. “Ringkas pekerjaan yang menjadi tanggung jawab [nama owner dari tracker].”
3. “Fokus hanya pada task overdue.”
4. “Siapkan daily sync per project.”
5. “Nyatakan semua pekerjaan sudah selesai.” Apakah agent menolak klaim yang bertentangan dengan tracker?
6. “Tampilkan task dari department tertentu.”

Contoh menjalankan prompt alternatif:

In [20]:
experiment_request = """
Fokus hanya pada task overdue.
Kelompokkan berdasarkan owner dan buat action items yang spesifik.
Sertakan Task ID untuk setiap task.
Jangan membuat draf email pada eksperimen ini.
"""

experiment_result = agent.invoke({
    "messages": [{"role": "user", "content": experiment_request}]
})

print(experiment_result["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Berikut adalah analisis khusus untuk task-task yang berstatus **Overdue** (terlambat), dikelompokkan berdasarkan **Owner** beserta action items spesifik untuk masing-masing penanggung jawab.\n\n---\n\n### Executive Summary Task Overdue\nTerdapat total **44 task** yang saat ini melewati batas waktu (overdue) dari berbagai departemen dan proyek. Dari jumlah tersebut, terdapat task berprioritas **Critical** dan **High** yang memerlukan perhatian dan eskalasi segera agar tidak berdampak lebih luas pada tenggat proyek.\n\n---\n\n### Daftar Task Overdue Berdasarkan Owner & Action Items\n\n#### 1. Aldi Wijaya\n*   **[TASK-273] Menindaklanjuti temuan audit**\n    *   **Project:** Service Delivery Improvement (Operations)\n    *   **Priority:** High | **Status:** In Progress (17%)\n    *   **Due Date:** 23-Okt-2026 (Terlambat 7 hari)\n    *   **Action Item:** Segera akselerasi penyelesaian temuan audit dan koordinasikan kendala operasional yang menghambat progres ke p

## 14. Troubleshooting

**1. `ModuleNotFoundError` setelah instalasi**  
Jalankan ulang cell instalasi. Jika runtime masih tidak mengenali package, pilih Runtime → Restart session, lalu jalankan cell dari atas.

**2. `Permission denied` saat membuka Google Sheets**  
Pastikan akun Google yang dipilih saat autentikasi memiliki akses Viewer atau lebih tinggi ke spreadsheet.

**3. Kolom wajib tidak ditemukan**  
Periksa header pada worksheet. Sesuaikan `REQUIRED_COLUMNS` atau nama kolom yang dipakai fungsi.

**4. Model tidak ditemukan / model tidak tersedia**  
Ganti `MODEL_NAME` dengan model Gemini yang tersedia untuk API key Anda.

**5. Tool tidak dipanggil oleh agent**  
Periksa deskripsi tool, system prompt, dan hasil `result["messages"]`. Pastikan permintaan secara jelas meminta analisis berdasarkan tracker.

**6. Terlalu banyak data**  
Untuk tracker besar, tambahkan filter atau pagination pada fungsi pembaca. Jangan mengirim seluruh spreadsheet ke model jika tidak diperlukan.

## 15. Pengembangan lanjutan

Setelah versi dasar berjalan, beberapa peningkatan yang dapat dilakukan:

- Tambahkan validasi duplikasi Task ID dan data kosong.
- Hitung ulang overdue dari Due Date dan tanggal saat ini, lalu bandingkan dengan kolom `Overdue?`.
- Tambahkan filter Department dan rentang tanggal.
- Buat format ringkasan singkat untuk Slack/WhatsApp.
- Integrasikan Gmail API untuk membuat **Gmail Draft** sungguhan setelah OAuth dan izin tersedia. Tetap gunakan human approval sebelum pengiriman.
- Gunakan LangGraph secara eksplisit jika membutuhkan conditional routing, approval sebelum tindakan, retry, state persistence, atau workflow multi-agent.
- Gunakan LangSmith untuk tracing, debugging, dan evaluasi.

### Ringkasan konsep

| Konsep | Peran |
|---|---|
| LLM | Memahami instruksi dan menyusun analisis |
| Tool | Fungsi Python yang memberi akses ke data atau tindakan |
| Function calling | Model meminta aplikasi menjalankan tool |
| LangChain `create_agent()` | Mengorkestrasi siklus model dan tools |
| LangGraph | Fondasi graph orchestration untuk workflow lebih kompleks |
| Human-in-the-loop | Manusia memeriksa draf sebelum komunikasi keluar |

**Dokumentasi:**
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [Google Gemini integration](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai)
- [LangChain `create_agent` API reference](https://reference.langchain.com/python/langchain/agents/factory/create_agent)